# Boussole Scolaire · Étape 01 · Collecte

**Défi 3** : prévoir les besoins et répartir au plus juste les enseignants et les salles de classe.

La collecte rassemble, sans rien modifier, les données utiles au défi parmi celles remises par les organisateurs. Elle s'organise autour de trois questions :

| Question | Données recherchées |
|---|---|
| Quelle est l'offre ? | enseignants (corps, âge, expérience), écoles, salles, état des salles, paillotes, équipements, logements de maître |
| Quelle sera la demande ? | effectifs par niveau, nouveaux entrants au CP1, projections de population, population par localité |
| Dans quel contexte ? | écoles fermées, rouvertes ou délocalisées ; déplacés internes ; référentiels géographiques |

Les sources sont déclarées dans `src/boussole/collecte/sources.json`, avec les motifs d'exclusion des fichiers non retenus. Le dossier des organisateurs est en lecture seule.

In [1]:
import json, sys
from pathlib import Path

PROJET = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJET / "src"))

import pandas as pd
from boussole.chemins import CODE_EQUIPE, LIEUX
from boussole.collecte import Collecte
from boussole.depots import deposer_collecte

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 90)
print("Données des organisateurs :", LIEUX.organisateurs)

Données des organisateurs : C:\Users\elias\OneDrive\Desktop\Education_competition_sentinelle\Competition_EducationBurkina


## 1. Sources déclarées

In [2]:
collecte = Collecte()
pd.DataFrame([{"source": s.id, "thème": s.theme, "niveau": s.niveau,
               "sélection": s.selection["type"] if not s.classeur else "classeur entier",
               "usage": s.usage} for s in collecte.sources])

,source,thème,niveau,sélection,usage
0,base_principale,offre et demande,région,classeur entier,Séries régionales d'élèves et d'écoles ; contrôle des totaux
1,dgess_principaux_chiffres,offre,province,section,Élèves par maître et par salle : diagnostic central
2,dgess_enseignants_corps,offre,province,section,"Maîtres chargés de cours, qualification, part des femmes"
3,dgess_enseignants_experience,offre,province,section,Structure par âge : départs à la retraite à anticiper
4,dgess_ecoles_statut,offre,province,section,"Écoles publiques, seules concernées par l'affectation"
5,dgess_ecoles_salles,offre,province,section,"Salles disponibles, public et privé"
6,dgess_taille_ecoles,offre,province,section,"Écoles incomplètes, besoin en salles"
7,dgess_equipement,offre,province,section,Conditions d'accueil des enseignants et des élèves
8,dgess_distance,accès,province,section,Accessibilité : contrainte du placement des écoles
9,dgess_salles_mise_en_service,offre,province,section,"Âge du parc de salles, rythme des constructions"


Trois modes de sélection :

- **section** (annuaires DGESS) : la page est retenue si son titre de section, numéroté en chiffres romains et écrit en majuscules, contient tous les mots-clés. Les numéros de page changent d'une édition à l'autre ; les titres de section restent stables.
- **tableaux** (annuaires de l'INSD) : la page est retenue si un titre de tableau contient les mots-clés ; les pages de suite sans titre sont rattachées au tableau précédent.
- **tout** : le fichier entier.

## 2. Exécution

In [3]:
collecte.executer()
resultats = collecte.ecrire()
registre = resultats["registre"]
print(f"{registre['source'].nunique()} sources · {len(registre)} fichiers · "
      f"{registre['nb_lignes'].astype(int).sum():,} lignes copiées".replace(",", " "))

26 sources · 205 fichiers · 262 829 lignes copiées


In [4]:
(registre.assign(nb_lignes=registre["nb_lignes"].astype(int))
 .groupby(["theme", "source"]).agg(fichiers=("fichier_origine", "count"), lignes=("nb_lignes", "sum"))
 .sort_index())

fichiers  lignes
theme            source                                        
accès            dgess_distance                       8    3515
demande          dgess_effectifs_niveau               8   21013
                 dgess_historique_effectifs           8    1323
                 dgess_nouveaux_entrants              8    7146
                 insd_projections                     2   30673
                 rgph_localites                       1   11862
                 rgph_synthese                        1    4925
offre            dgess_ecoles_salles                  8    2322
                 dgess_ecoles_statut                  8     739
                 dgess_enseignants_corps              8   12660
                 dgess_enseignants_experience         8   12020
                 dgess_equipement                     8    4666
                 dgess_etat_salles                    8    7622
                 dgess_logements_maitre               8    2613
                 dgess_principaux_chiffres            8    1869
                 dgess_salles_mise_en_service         8    2709
                 dgess_taille_ecoles                  8    4226
                 insd_annuaires_nationaux            10    1973
offre et demande base_principale                      1    1568
                 dgess_tableaux_de_bord              10   15436
                 insd_communes_primaire              10   13222
référentiel      codab_communes                       1     352
                 codab_provinces_regions              2      66
                 ecoles_osm                           1   11974
sécurité         deplaces_internes                   30   65635
                 esu_rapports                        24   20700

## 3. Contrôle de couverture des annuaires

Chaque section doit être trouvée dans chacune des huit éditions (2017-2018 à 2024-2025), avec un volume comparable d'une année à l'autre.

In [5]:
couverture = resultats["couverture"]
annuaires = couverture[couverture["source"].str.startswith("dgess_")].copy()
annuaires["nb_lignes"] = annuaires["nb_lignes"].astype(int)
annuaires.pivot_table(index="source", columns="periode", values="nb_lignes", aggfunc="sum", fill_value=0)

periode,2017-2018,2018-2019,2019-2020,2020-2021,2021-2022,2022-2023,2023-2024,2024-2025
source,,,,,,,,
dgess_distance,440,440,440,440,440,435,440,440
dgess_ecoles_salles,293,293,293,293,293,283,287,287
dgess_ecoles_statut,89,96,96,96,96,88,89,89
dgess_effectifs_niveau,2536,2544,2776,2772,2773,2536,2536,2540
dgess_enseignants_corps,1585,1585,1585,1585,1585,1565,1585,1585
dgess_enseignants_experience,1485,1525,1525,1525,1525,1465,1485,1485
dgess_equipement,585,585,585,585,583,575,584,584
dgess_etat_salles,954,952,954,954,954,946,954,954
dgess_historique_effectifs,164,169,169,169,169,161,161,161


In [6]:
couverture[couverture["alerte"] != ""]

,source,fichier_origine,periode,pages,nb_pages,nb_lignes,alerte


## 4. Bilan des fichiers remis

Chaque fichier source remis par les organisateurs est soit retenu (avec les sources qui l'utilisent), soit écarté avec un motif écrit.

In [7]:
bilan = resultats["bilan"]
bilan["statut"].value_counts()

statut
retenu    89
écarté    84
Name: count, dtype: int64

In [8]:
bilan[bilan["statut"] == "écarté"].groupby("motif").size().sort_values(ascending=False).rename("fichiers").to_frame()

,fichiers
motif,
Éducation non formelle : hors du champ (enseignement primaire formel),22
"Conditions de vie, pauvreté, santé, emploi : déterminants de la demande, sans lien avec l'offre d'enseignants et de salles",14
"Synthèses et dépliants : mêmes données que les annuaires retenus, sous forme résumée",12
"Géométries et métadonnées cartographiques : utilisées pour les cartes, sans tableau à collecter",9
Résultats généraux et analyses du RGPH sans donnée sur l'offre scolaire,7
Préscolaire : hors du champ,6
Documentation du challenge et du Data Pack,5
"Post-primaire, secondaire et supérieur : la répartition étudiée porte sur le primaire",3
"Documentation du RGPH : définitions citées, aucune donnée à extraire",3


## 5. Aperçu : la table centrale du diagnostic

Le tableau I.1 des annuaires donne, par province et par statut, les écoles, les salles de classe, les élèves, les groupes pédagogiques et les maîtres. C'est de lui que viennent les ratios élèves par maître et élèves par salle. Extrait brut de l'édition 2024-2025 :

In [9]:
chiffres = collecte.extraits["dgess_principaux_chiffres"]
apercu = chiffres[chiffres["origine_periode"] == "2024-2025"]
apercu[["page", "ligne"] + [f"c{i}" for i in range(1, 10)]].head(14)

,page,ligne,c1,c2,c3,c4,c5,c6,c7,c8,c9
1635,12,1,BF/MEBAPLN/SG/DGESS,Année Scolaire 2024-2025,,,,,,,
1636,12,2,I. LES PRINCIPAUX CHIFFRES,,,,,,,,
1637,12,3,"I.1 NOMBRE D'ECOLES, DE SALLES DE CLASSE, D'ELEVES DE GROUPES PEDAGOGIQUES ET DE MAITR...",,,,,,,,
1638,12,4,Provinces,Statut,Nombre,Nombre de,Nombre de,Nombre de,Nombre,Ratio,
1639,12,5,d'écoles,salles de,groupes,maîtres,d'élèves,élèves/maître,,,
1640,12,6,classe,pédagogiq.,chargés de,,,,,,
1641,12,7,cours,,,,,,,,
1642,12,8,BOUCLE DU MOUHOUN,,,,,,,,
1643,12,9,BANWA,Public,75,404,377,374,23 416,"62,6",
1644,12,10,Privé,22,85,103,89,5 240,"58,9",,


## 6. Dépôt au format du challenge

In [10]:
depot = deposer_collecte(version=1)
print("Code équipe :", CODE_EQUIPE, "· fichiers déposés :", len(depot))
depot[["fichier", "role"]]

Code équipe : EXX · fichiers déposés : 30


,fichier,role
0,EXX_01-collecte_D3_engineer_registre-des-sources_v1.csv,engineer
1,EXX_01-collecte_D3_engineer_bilan-sources-retenues-ecartees_v1.csv,engineer
2,EXX_01-collecte_D3_engineer_inventaire-fichiers-organisateurs_v1.csv,engineer
3,EXX_01-collecte_D3_engineer_controle-couverture-pages_v1.csv,engineer
4,EXX_01-collecte_D3_engineer_base-principale-dgess-region-2016-2025_v1.csv,engineer
5,EXX_01-collecte_D3_engineer_ecoles-salles-maitres-eleves-province-2017-2025_v1.csv,engineer
6,EXX_01-collecte_D3_engineer_enseignants-corps-fonction-province-2017-2025_v1.csv,engineer
7,EXX_01-collecte_D3_engineer_enseignants-age-experience-province-2017-2025_v1.csv,engineer
8,EXX_01-collecte_D3_engineer_ecoles-par-statut-province-2017-2025_v1.csv,engineer
9,EXX_01-collecte_D3_engineer_ecoles-salles-public-prive-province-2017-2025_v1.csv,engineer


## Limites

- Les extraits gardent la mise en page des PDF : la remise en forme est l'objet de l'étape 02.
- Aucune donnée par école ni fichier des enseignants : le diagnostic descend à la province et, pour 10 régions sur 13, à la commune (annuaires régionaux de l'INSD).
- Les rapports ST-ESU les plus récents ont été lus par reconnaissance de caractères : leurs chiffres seront contrôlés au nettoyage.